In [1]:
!pip install wandb

In [3]:
!wandb login

wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kapoorprateek91 (kapoorprateek91-rsa) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [6]:
!wandb login --relogin


wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [7]:
import wandb
print(f'W&Bversion: {wandb.__version__}')
print('Authentication status: logged in' if wandb.api.api_key else 'Not logged in')

W&Bversion: 0.25.1
Authentication status: logged in


In [10]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

#Generate synthetic real estate dataset
np.random.seed(42)
n = 200

sqft = np.random.normal(1500,400,n)
bedrooms = np.random.randint(1,6,n)
age = np.random.normal(20,10,n).clip(1)
distance_to_city = np.random.exponential(5,n)
noise = np.random.normal(0,20000,n)

price = (150 * sqft + 15000 * bedrooms - 1000 * age - 5000 * distance_to_city + noise)
df = pd.DataFrame({'sqft':sqft, 'bedrooms':bedrooms, 'age':age, 'distance_to_city':distance_to_city, 'price':price})

x = df.drop('price', axis = 1)
y = df['price']

print('dataset shape:', df.shape)
print(df.head())
print(df.describe())

dataset shape: (200, 5)
          sqft  bedrooms        age  distance_to_city          price
0  1698.685661         2  32.049229          1.642375  237528.113635
1  1444.694280         2   8.033395          7.337369  197399.581630
2  1759.075415         1  27.706783          1.035390  262254.857600
3  2109.211943         1  26.670561          1.955439  276125.759530
4  1406.338650         1  13.008534          2.770723  203781.689649
              sqft    bedrooms         age  distance_to_city          price
count   200.000000  200.000000  200.000000        200.000000     200.000000
mean   1483.691614    3.035000   20.603304          4.700595  222681.459890
std     372.401566    1.457531    9.836254          4.586707   69217.170502
min     452.101958    1.000000    1.000000          0.023214   59181.206580
25%    1217.948930    2.000000   14.543031          1.294452  176169.828129
50%    1498.323246    3.000000   20.121975          3.237268  221127.477373
75%    1700.340989    4.000000

In [15]:
run = wandb.init(
    project = "demo-1",
    name = 'ridge-alpha-1.0',
    config = {
        "model": "Ridge",
        "alpha": 1.0,
        "cv_folds": 5,
        "scaler": "StandardScaler",
        "dataset_size": n
    }
)
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha = 1.0))
])
#Evaluate using 5-fold cross-validation
scores = cross_val_score(pipe, x, y, cv=5, scoring='r2')
wandb.log({
    'mean_r2': scores.mean(),
    'std_r2': scores.std(),
    'min_r2': scores.min(),
    'max_r2': scores.max()
})
print(f'Ridge (alpha=1.0): R2 = {scores.mean():.4f} +/- {scores.std():.4f}')
print(f'View this run at: {run.get_url()}')
#Close the run
wandb.finish()

Ridge (alpha=1.0): R2 = 0.9031 +/- 0.0070
View this run at: https://wandb.ai/kapoorprateek91-rsa/demo-1/runs/deyd3xij


max_r2,▁
mean_r2,▁
min_r2,▁
std_r2,▁
max_r2,0.91606
mean_r2,0.90311
min_r2,0.895
std_r2,0.00701


In [18]:
models = {
    'Ridge': Ridge,
    'Lasso': Lasso,
    'ElasticNet': ElasticNet
}
alphas = [0.01, 0.1, 1.0, 10.0]
print('Tracking 12 model configurations')
for model_name, ModelClass in models.items():
    for alpha in alphas:
        run = wandb.init(
          project = "demo-2",
          name = f'{model_name.lower()}-alpha-{alpha}',
          config = {
              "model": model_name,
              "alpha": alpha,
              "cv_folds": 5,
              "scaler": "StandardScaler",
              "dataset_size": 200
          }
        )
        #Build and configure model
        if model_name == 'ElasticNet':
            model = ModelClass(alpha=alpha, l1_ratio=0.5)
        else:
            model = ModelClass(alpha=alpha)
        #Create pipeline and evaluate
        pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('model', model)
        ])
        scores = cross_val_score(pipe, x, y, cv=5, scoring='r2')
        #Log comprehensive metrices
        wandb.log({
            'mean_r2': scores.mean(),
            'std_r2': scores.std(),
            'min_r2': scores.min(),
            'max_r2': scores.max(),
            'cv_range': scores.max() - scores.min()
        })
        print(f"{model_name:12} (alpha={alpha:>5}): R2 = {scores.mean():.4f} +/- {scores.std():.4f}")
        wandb.finish()
print("All 12 logged to W&B")

Tracking 12 model configurations


Ridge        (alpha= 0.01): R2 = 0.9030 +/- 0.0073


cv_range,▁
max_r2,▁
mean_r2,▁
min_r2,▁
std_r2,▁
cv_range,0.02164
max_r2,0.91682
mean_r2,0.90302
min_r2,0.89517
std_r2,0.00728


Ridge        (alpha=  0.1): R2 = 0.9030 +/- 0.0073


cv_range,▁
max_r2,▁
mean_r2,▁
min_r2,▁
std_r2,▁
cv_range,0.02159
max_r2,0.91675
mean_r2,0.90303
min_r2,0.89516
std_r2,0.00725


...

ElasticNet   (alpha=  1.0): R2 = 0.8074 +/- 0.0185


cv_range,▁
max_r2,▁
mean_r2,▁
min_r2,▁
std_r2,▁
cv_range,0.05108
max_r2,0.84353
mean_r2,0.80743
min_r2,0.79245
std_r2,0.0185


ElasticNet   (alpha= 10.0): R2 = 0.2702 +/- 0.0182


cv_range,▁
max_r2,▁
mean_r2,▁
min_r2,▁
std_r2,▁
cv_range,0.05086
max_r2,0.28651
mean_r2,0.27018
min_r2,0.23565
std_r2,0.01824


All 12 logged to W&B


In [22]:
# W&B + Optuna integration
!pip install optuna
!pip install optuna-integration[wandb]


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 2.0 MB/s eta 0:00:00


In [24]:
import optuna
from optuna.integration.wandb import WeightsAndBiasesCallback
wandb_callback = WeightsAndBiasesCallback(
    metric_name = 'mean_r2',
    wandb_kwargs = {"project":"optuna_sweep"}
)
def objective(trial):
  alpha = trial.suggest_float('alpha', 0.001, 10.0, log=True)
  l1_ratio = trial.suggest_float('l1_ratio', 0.0, 1.0)

  pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', ElasticNet(alpha=alpha, l1_ratio=l1_ratio))
  ])
  scores = cross_val_score(pipe, x, y, cv=5, scoring='r2')
  return scores.mean()
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30, callbacks=[wandb_callback])

wandb.finish()

[I 2026-03-27 20:21:53,397] A new study created in memory with name: no-name-3a603b46-2cca-40f2-a5ac-6fcc9f40a5df
[I 2026-03-27 20:21:53,448] Trial 0 finished with value: 0.903028859051847 and parameters: {'alpha': 0.0012263898302312384, 'l1_ratio': 0.5714178024663461}. Best is trial 0 with value: 0.903028859051847.
[I 2026-03-27 20:21:53,493] Trial 1 finished with value: 0.9030486768782344 and parameters: {'alpha': 0.013404727416278754, 'l1_ratio': 0.8817056433915947}. Best is trial 1 with value: 0.9030486768782344.
[I 2026-03-27 20:21:53,534] Trial 2 finished with value: 0.5362494020069906 and parameters: {'alpha': 2.4715386023430055, 'l1_ratio': 0.2828203114800245}. Best is trial 1 with value: 0.9030486768782344.
[I 2026-03-27 20:21:53,576] Trial 3 finished with value: 0.9030247649049574 and parameters: {'alpha': 0.001786651195011864, 'l1_ratio': 0.8211146944818977}. Best is trial 1 with value: 0.9030486768782344.
[I 2026-03-27 20:21:53,619] Trial 4 finished with value: 0.8944625308

alpha,▁▁▄▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁█▁▁▁▁▁▁▁▁▁▁▁
l1_ratio,▅▇▃▇▅▅▃▃▂▇▆▁▄▄▄▆▄█▅▆▁▆▆▅▃▄▃▃▂▄
mean_r2,██▃███████████████▁███████████
alpha,0.00125
l1_ratio,0.50206
mean_r2,0.90303
